# Compartment calling and evaluation tutorial

This tutorial follows the Compartment workflow from raw and imputed contact
maps to compartment tracks, cell-number trajectories and task-specific
evaluation. It covers **100 kb and 500 kb**, single cells and pseudo-bulk
samples, and the Self-PCA, CpG, Ref-PCA and scGHOST branches.

This is a reading tutorial. Commands and configuration examples refer to the
companion analysis implementation; they are displayed as text. No
completed local Compartment result table is embedded here. The final section
shows the output files and their meanings.

```text
Original metadata + raw / scHiCluster / Higashi contact inputs
                 |
    Single cells and exact pseudo-bulk member lists
                 |
    CpG / Self-PCA / Ref-PCA / scGHOST (500 kb)
                 |
    Native compartment tracks + row identities
                 |
    Boundary F1 | track correlation | saddle strength | cell clustering
```


Use the [shared path setup](tutorials.ipynb#Shared-path-prefix) for the commands below:

```bash
source tutorials/paths.sh
cd "${BENCHMARK_ROOT}/code"
```

```python
from pathlib import Path
import os

BENCHMARK_ROOT = Path(os.environ["BENCHMARK_ROOT"])
```


<a id="Inputs-and-evaluation-design"></a>

## 1. Inputs and evaluation design


<a id="1.-Define-the-evaluated-conditions"></a>

### 1.1. Define the evaluated conditions

The input modality and the compartment caller are separate choices:

| Resolution | Compartment callers | Input modalities | Analysis levels | Conditions per dataset |
|---|---|---|---|---:|
| 100 kb | Self-PCA, CpG, Ref-PCA | Raw, scHiCluster, Higashi | Single cell, pseudo-bulk | 18 |
| 500 kb | Self-PCA, CpG, Ref-PCA, scGHOST | Raw, scHiCluster, Higashi | Single cell, pseudo-bulk | 24 |

For example, **Ref-PCA + Higashi** means applying the Ref-PCA caller to the
Higashi input modality. The evaluator uses `ref_pca` for the caller and
`higashi` for the input, preserving that distinction.

The full matrix contains 42 evaluated conditions across the two resolutions.
There are 39 native source conditions; the three scGHOST pseudo-bulk
conditions are derived from their single-cell outputs.


<a id="2.-Prepare-identities-and-exact-sample-membership"></a>

### 1.2. Prepare identities and exact sample membership

Use the original metadata labels and the original cell-number subsets when
describing an existing experiment. The main input contracts are:

| Input | Required fields or contents |
|---|---|
| `metainfo.tsv` | Headered `cell_id`, `cell_type` |
| `samples.tsv` | `sample_id`, `cell_type`, `cell_num`, `members_path` |
| Member list | Headered `cell_id`, one selected biological ID per row |
| Genome sizes | Chromosome names and lengths for the corresponding genome |
| Contact table | Headered `cell_id`, `path` for the evaluator's Cooler adapter |
| Native row-order sidecar | Headered `cell_id`, in the actual output row order |
| Existing cell-cluster assignments | Headered `cell_id`, `cluster` |

Native CpG/Self-PCA calling uses its own **headerless** two-column cell table.
That calling table and the evaluator's headered contact table have different
contracts. Pseudo-bulk caller rows use the IDs from `samples.tsv`.

An example sample-manifest record is:

```text
sample_id   cell_type   cell_num   members_path
ODC.100     ODC         100        members/ODC.100.tsv
```

The actual TSV uses tab separators. Native HDF5 numeric rows and scGHOST
rows require their original ID-order sidecars. Metadata order is used only
when it is the verified native row order.


<a id="3.-Arrange-the-cell-number-gradient-and-raw-reference"></a>

### 1.3. Arrange the cell-number gradient and raw reference

For structural evaluation, the default configuration selects up to the two
largest cell types supporting a series beginning at 100 cells. Each selected
type supplies **100, 200, 300, …, all cells**, including the exact all-cell
endpoint once. The same member lists define each condition across callers,
input modalities and resolutions.

The evaluation reference for each selected cell type is:

```text
All raw cells of that type → raw pseudo-bulk → Self-PCA compartment track
```

Generate that reference separately at 100 kb and 500 kb. It is the track
selected by the evaluator's `pca / raw / pseudo_bulk` condition at the
all-cell endpoint. Smaller pseudo-bulk conditions are compared with this
same cell-type reference at the corresponding resolution.

For new input subsets, the existing helper has this interface:

```bash
python compartment/tool/evaluation/prepare_compartment_inputs.py \
  --metadata ${COMPARTMENT_METADATA} --seed 42 --output-dir ${COMPARTMENT_PREPARED_DIR}
```

That seed describes a new selection. For an existing manuscript condition,
retain its recorded members instead of regenerating them.


<a id="Compartment-calling"></a>

## 2. Compartment calling


<a id="4.-Call-CpG-based-compartments"></a>

### 2.1. Call CpG-based compartments

CpG calling combines contact weights with a genome- and resolution-matched
CpG-density vector. After diagonal removal and its native normalization, the
core calculation in [compartment/tool/CpG/calling.py](https://github.com/rawandy2001-design/benchmark/blob/02e6c9eb2bb9c4b549b197b4b47fd5d9ae34ada8/compartment/tool/CpG/calling.py) is:

```python
# matrix is the normalized sparse chromosome matrix from the caller.
# cpg_ratio is the matched per-bin CpG-density Series.
comp = matrix.dot(cpg_ratio.values[:, None])[:, 0]
```

The result is a contact-weighted CpG-density profile. At the same genome and
resolution, samples share the same CpG profile. The 100-kb and 500-kb conditions
read separate profiles.

The native 100-kb calling interface is:

```bash
python compartment/tool/CpG/calling.py \
  --dataset Lee --cell_table_path ${COMPARTMENT_CELL_TABLE} \
  --output_prefix ${COMPARTMENT_CPG_PREFIX} \
  --resolution 100000 --mode cool --cpu 10
```

The legacy caller selects the hg38 profile for dataset tokens `Lee` and
`Tian`; other tokens select mm10. Its CpG and chromosome-size resource paths
are configured in the source. Align those resources before running it in
the this folder. Output is `<output_prefix>.compartment.nc`.


<a id="5.-Call-Self-PCA-compartments"></a>

### 2.2. Call Self-PCA compartments

Self-PCA fits each target matrix independently. Its default workflow applies
`sqrtvc` normalization, distance-dependent expectation correction, a bin-to-bin
correlation matrix and two-component PCA. Saddle contrast selects between the
candidate components, and CpG information calibrates the sign.

```bash
python compartment/tool/Self-PCA/calling.py \
  --dataset Lee --cell_table_path ${COMPARTMENT_CELL_TABLE} \
  --output_prefix ${COMPARTMENT_PCA_PREFIX} \
  --resolution 100000 --mode cool --cpu 10 \
  --norm_type sqrtvc --exp_type dicrete
```

`dicrete` is the exact option spelling in this implementation. The same
caller is applied separately to the configured raw or imputed sample maps.
The usual output variable is `compartment`; optional double-save output
retains both candidate components and their selection records.

The CpG and Self-PCA 500-kb branches currently coarsen their 100-kb matrix
inputs internally. Keep that input convention explicit when preparing their
native 500-kb calls. Evaluation then reads the resulting 500-kb tracks.


<a id="6.-Call-Ref-PCA-compartments"></a>

### 2.3. Call Ref-PCA compartments

Ref-PCA uses the bundled Higashi analysis routines. It constructs a merged
raw background, masks bins, splits chromosome arms using cytoband information
and fits PCA models on that background. Target raw or imputed matrices are
projected onto those fitted axes. CpG calibration and quantile/z-score
versions are retained by the native output.

The two original script interfaces are:

```bash
python compartment/tool/Ref-PCA/calling_single_cell.py \
  --config ${COMPARTMENT_CONFIG} --data_type higashi \
  --calib --output scCompartment.Higashi.100K.hdf5

python compartment/tool/Ref-PCA/calling_bulk.py \
  --config ${COMPARTMENT_CONFIG} --temp ${COMPARTMENT_TEMP_DIR} \
  --bulk bulk100 --data_type higashi --calib \
  --output scCompartment.Higashi.bulk100.100K.hdf5
```

The single-cell script reads the configured raw cell bank. The bulk script
reads the bank for the selected `bulk` condition. Their legacy data paths and
resolution-matched native resources must correspond to that condition. An
optional `bulk_path` selects an external merged matrix for PCA fitting.

This fitted projection background is distinct from the raw all-cell Self-PCA
track used for **evaluation** in section 1.3. The two roles remain separate in
the tutorial and input manifest.


<a id="7.-Supply-native-scGHOST-outputs-at-500-kb"></a>

### 2.4. Supply native scGHOST outputs at 500 kb

Generate scGHOST outputs in its original environment using the
[official tutorial](https://github.com/ma-compbio/scGHOST/blob/3271dfa1b441fcfe2ed9aad83888b78922aba91e/tutorial.ipynb).
The existing adapter follows the pinned native
[clustering output](https://github.com/ma-compbio/scGHOST/blob/3271dfa1b441fcfe2ed9aad83888b78922aba91e/modules/clustering.py)
and accepts either:

- `labels.pkl`, `cropped_indices.pkl` and a native row-order cell-ID sidecar;
- official four-column BED exports, listed by biological cell ID.

The evaluator's numerical mapping is:

| Native label | Subcompartment | Evaluation value |
|---|---|---:|
| 0 | A1 | 1 |
| 1 | A2 | 1 |
| 2 | B1 | −1 |
| 3 | B2 | −1 |
| 4 | B3 | −0.5 |

`cropped_indices.pkl` places these values on their original 500-kb genomic
bins. Missing bins remain missing. Pseudo-bulk tracks are the per-bin means
of exactly the member cells listed for each sample. These genomic labels
are separate from the cell-cluster assignments evaluated later.


<a id="8.-Declare-native-tracks-and-contact-sources"></a>

### 2.5. Declare native tracks and contact sources

The source manifest identifies dataset, resolution, caller, input modality,
analysis level, file format and path. Supported profile formats include:

| Format | Native or standardized content |
|---|---|
| `netcdf` | CpG/Self-PCA scores and genomic coordinates |
| `higashi_hdf5` | Ref-PCA `compartment/cell_N` and `compartment/bin/` |
| `scghost_native` | Native labels, cropped indices and row IDs |
| `scghost_bed` | Cell-ID/BED manifest |
| `npz` | IDs, coordinates, scores and declared resolution |
| `long_tsv` | `cell_id,chrom,start,end,score` |

For example, these are individual source specifications using the evaluator's
existing field names. An actual full manifest supplies all native conditions:

```json
{
  "dataset": "Lee2019",
  "resolution": 100000,
  "method": "ref_pca",
  "input": "higashi",
  "level": "single_cell",
  "format": "higashi_hdf5",
  "path": "/data/calls/scCompartment.Higashi.100K.hdf5",
  "cell_ids_path": "/data/calls/Higashi.100K.cell_ids.tsv",
  "clustering": {
    "algorithm": "native_labels",
    "path": "/data/clusters/ref_pca.higashi.100K.tsv",
    "label_column": "cluster"
  }
}
```

Contact sources are declared separately. For Cooler input:

```json
{
  "dataset": "Lee2019",
  "resolution": 100000,
  "input": "raw",
  "level": "single_cell",
  "format": "cool_table",
  "path": "/data/raw/contact_table.tsv"
}
```

The evaluator also accepts chromosome-specific sparse NPY or matrix NPZ
sources with explicit IDs. If a pseudo-bulk contact source is omitted, it
sums the exact member matrices from the corresponding single-cell source.
CpG and Ref-PCA profiles are centered per cell/sample and chromosome by
default; the manifest can set this explicitly.


<a id="Structural-and-cell-level-evaluation"></a>

## 3. Structural and cell-level evaluation


<a id="9.-Compare-boundary-locations-and-compartment-tracks"></a>

### 3.1. Compare boundary locations and compartment tracks

The evaluator compares each target track with its selected cell-type raw
all-cell reference on finite matched bins. The configured denoising step
operates before sign-change boundaries are identified.

Boundary matching is one-to-one with **one-bin tolerance**. The exports
include precision, recall, F1 and boundary counts. Missing bins split genomic
runs, so they do not create a boundary across an unobserved interval.

Track correlation is the Pearson correlation of the aligned continuous
compartment values. Both metrics are recorded separately for each chromosome,
sample and condition.


<a id="10.-Calculate-saddle-strength-on-the-matched-contact-map"></a>

### 3.2. Calculate saddle strength on the matched contact map

Each compartment profile is paired with its own input modality's contact
matrix, matched by cell or sample ID. The evaluator removes the diagonal,
calculates the legacy distance-dependent raw observed/expected matrix and
sets its diagonal to one.

Bins are ordered into up to 50 score quantiles. The top and bottom 20% of
occupied groups define the A and B tails. With fewer groups from repeated
scores, the calculation uses the occupied groups explicitly.

```text
saddle strength = (sum(AA) + sum(BB)) / (sum(AB) + sum(BA) + 1e-8)
```

The contact adapter can sum 100-kb contact blocks to 500 kb. Compartment
profiles themselves come from their corresponding 500-kb calling outputs;
they are aligned rather than averaged from 100-kb score tracks.


<a id="11.-Evaluate-existing-cell-cluster-assignments"></a>

### 3.3. Evaluate existing cell-cluster assignments

The default mode is `native_labels`. It reads the selected workflow's final
cell groups and compares them with the original metainfo using ARI, AMI
and V-measure. Calinski–Harabasz is calculated from the genome-wide
compartment features and those groups.

```json
{
  "algorithm": "native_labels",
  "path": "/data/clusters/pca.raw.100K.tsv",
  "label_column": "cluster"
}
```

These groups are per-cell assignments, distinct from the A/B labels at
genomic bins. Structural cell-type selection and the clustering population
are separate configuration choices; by default clustering uses the configured
metadata cells.

The implementation also permits an explicitly selected new K-means analysis.
The original-assignment route above is the one described here.


<a id="Run-the-workflow-and-read-outputs"></a>

## 4. Run the workflow and read outputs


<a id="12.-Validate-and-evaluate-the-configured-workflow"></a>

### 4.1. Validate and evaluate the configured workflow

The original templates are
[compartment/tool/evaluation/configs/benchmark.example.json](https://github.com/rawandy2001-design/benchmark/blob/02e6c9eb2bb9c4b549b197b4b47fd5d9ae34ada8/compartment/tool/evaluation/configs/benchmark.example.json) and
`profiles.example.tsv` in this folder. Fill them with the
native track sources, members, contact tables and cluster files described
above. Paths in JSON resolve relative to that JSON; paths in the source
manifest resolve relative to the manifest.

The main configuration selects:

```json
{
  "resolutions": [100000, 500000],
  "clustering": {"algorithm": "native_labels"},
  "options": {
    "step": 100,
    "denoise": true,
    "boundary_tolerance": 1,
    "require_complete_matrix": true,
    "require_contacts": true,
    "saddle_quantiles": 50,
    "saddle_tail_fraction": 0.2
  }
}
```

This excerpt shows shared options; the full configuration also declares
`datasets`, `profile_manifest`, `contacts` and `output_dir`.
The direct command-line entry is:

```bash
python compartment/tool/evaluation/run_compartment_benchmark.py \
  --config ${COMPARTMENT_BENCHMARK_CONFIG} --validate-only

python compartment/tool/evaluation/run_compartment_benchmark.py \
  --config ${COMPARTMENT_BENCHMARK_CONFIG}
```

Validation reads the configured identities, profiles, complete gradients,
references and contact definitions. Evaluation then writes the separate
structural and clustering tables.


<a id="13.-Read-the-output-files"></a>

### 4.2. Read the output files

The evaluation produces the following result groups:


| Output | Meaning |
|---|---|
| Chromosome table | Boundary metrics, track correlation, saddle values and observed-bin counts |
| Sample table | Equal-chromosome mean for each cell or pseudo-bulk sample |
| Structural summary | Equal-cell mean for single-cell conditions; separate pseudo-bulk cell-number conditions |
| Clustering table | ARI, AMI, V-measure, CHI and assignment parameters |
| Condition table | Dataset, resolution, caller, modality and analysis level actually evaluated |
| Run manifest | Resolved sources, exact members, versions and calculation conventions |

These are output contracts, not newly generated result files in this reading
package. Each metric retains its own table columns.


### 4.3. Compare Compartment versions

Open the [current Compartment notebook](compartment.ipynb) or [current webpage](compartment.html) to read the original-source notebook version.


<a id="Software-and-implementation"></a>

### 4.4. Software and implementation

| Stage | Software context |
|---|---|
| CpG/Self-PCA calling | NumPy, pandas, SciPy, scikit-learn, Cooler, Xarray and HDF support |
| CpG resource generation | Matching genome FASTA and bedtools |
| Ref-PCA calling | Bundled Higashi analysis routines, Cooler, h5py, joblib and native configuration |
| scGHOST generation | The official scGHOST runtime |
| Shared evaluation | NumPy, pandas, SciPy, scikit-learn, h5py; Cooler for contact tables |

The implementation is under `compartment/tool/` in this folder.
Its calling and evaluation guides specify the native paths and file contracts.
This tutorial records those interfaces without starting native jobs.

Return to the [current Compartment version](compartment.ipynb).

[Environment instructions](https://github.com/rawandy2001-design/benchmark/blob/02e6c9eb2bb9c4b549b197b4b47fd5d9ae34ada8/compartment/ENVIRONMENT.md) · [Repository guide](../README.md)
